# DumpWatch 2.0 | Task A

Use a CPU or GPU Colab runtime. GPU is useful for frozen ResNet-18 feature extraction. Authorize the read-only Drive mount before proceeding. The script never writes to the dataset folders.

In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get("DUMPWATCH_PACKAGE", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "project_setup.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within the unpacked code folder or set DUMPWATCH_PACKAGE to it. See SETUP.md.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from project_setup import project_paths, new_output, enable_demo_vendor
P = project_paths()
enable_demo_vendor()
DATA_ROOT = P["task_a_root"]
RUNNER = ROOT / "Reproduction/Task_A/task_a.py"
print("Task A training notebook. Full fitting is optional and was NOT rerun during recovery.")
for part, count in [("TRAINING_SET", 400), ("VALIDATION_SET", 100)]:
    directory = DATA_ROOT / part
    actual = [len(list((directory / "videos").glob("*.mp4"))), len(list((directory / "labels").glob("*.json")))]
    if actual != [count,count]: raise RuntimeError(f"{directory}: expected {count} MP4/JSON pairs, found {actual}")
    print(part, actual)


## Pipeline source

This cell writes the exact standalone runner into the temporary Colab runtime.

In [ ]:
assert RUNNER.is_file(), RUNNER
print("Using the unchanged original training runner:", RUNNER)


## Smoke test

Uses only a small subset of TRAINING_SET, verifies decoding, trains small models and tests predictions. These numbers are operational diagnostics.

In [ ]:
import subprocess, sys, os, shutil
RUN_BASE = new_output("task_a_training")
CACHE = RUN_BASE / "features"
# Seed the original torchvision weight cache with the exact bundled encoder.
os.environ["TORCH_HOME"] = str(RUN_BASE / "torch")
encoder_dir = RUN_BASE / "torch/hub/checkpoints"
encoder_dir.mkdir(parents=True)
shutil.copy2(ROOT / "Demo/models/resnet18-f37072fd.pth", encoder_dir / "resnet18-f37072fd.pth")
env = os.environ.copy()
if sys.platform == "win32" and sys.version_info[:2] == (3, 14):
    env["PYTHONPATH"] = str(ROOT / "Demo/vendor") + os.pathsep + env.get("PYTHONPATH", "")
def run_task_a(stage):
    dest = RUN_BASE / stage
    subprocess.run([sys.executable, "-B", "-u", str(RUNNER), "--root", str(DATA_ROOT),
                    "--out", str(dest), "--cache", str(CACHE), "--mode", stage], check=True, env=env)
    return str(dest)
SMOKE = run_task_a("smoke")


## Full run

The script validates all 400 training-source clips before fitting, creates an internal scene-proxy grouped development split, and freezes models before opening the 100-video holdout. Use CPU only if you can leave the session connected for the embedding step.

In [ ]:
assert Path(SMOKE, "development_metrics.json").is_file()
RESULTS = run_task_a("full")

## Save the evidence

Download the result ZIP before the runtime disconnects. It contains no source videos.

In [ ]:
import shutil
archive = shutil.make_archive(RESULTS, "zip", RESULTS)
print("Training output archive:", archive)
